In [8]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.transform import Affine
from tqdm import tqdm
import math, os
from matplotlib import pyplot as plt
from pathlib import Path

In [2]:
country_raster_1km_ = rio.open(r'../results/country-rasters/country-raster-eck4-1km.tif')

### mammals

In [3]:
mammals_ = gpd.read_file('../data/species-range-maps/MAMMALS/MAMMALS.shp')

In [4]:
mammals = mammals_[(mammals_['presence'] == (1 or 2 or 3)) & (mammals_['origin'] == (1 or 2 or 6)) & (mammals_['terrestria'] == 'true')]

In [5]:
mammals_eck4 = mammals.to_crs('ESRI:54012')

In [6]:
species_names = mammals_eck4['sci_name'].unique().tolist()
species_category = [mammals_eck4[mammals_eck4['sci_name'] == species]['category'].values[0] for species in species_names]

range_index_mammals = pd.DataFrame([], columns=['name', 'row_min', 'row_max', 'col_min', 'col_max'])
range_index_mammals['name'] = species_names
range_index_mammals['category'] = species_category

In [7]:
Path(r'../results/species-range-rasters/mammals').mkdir(parents=True, exist_ok=True)

In [9]:
for i in tqdm(range_index_mammals.index):
    a_species = range_index_mammals.loc[i, 'name']
    try:
        # print(a_species)
        a_species_range = mammals_eck4[mammals_eck4['sci_name'] == a_species].geometry 
        a_species_polygons = [(a_geometry, 1) for a_geometry in a_species_range]
        reprojected_raster = rasterize(
            shapes=a_species_polygons, 
            out_shape=country_raster_1km_.shape, 
            transform=country_raster_1km_.transform,
            all_touched=True,
            fill=0, 
            dtype=np.uint8,
        )
        rows, cols = np.where(reprojected_raster == 1)
        row_min, row_max = rows.min(), rows.max()
        col_min, col_max = cols.min(), cols.max()
        the_slice = reprojected_raster[row_min:row_max+1, col_min:col_max+1]

        output_path = os.path.join('..', 'results', 'species-range-rasters', 'mammals', f'{a_species}.tif')
        new_transform = country_raster_1km_.transform * Affine.translation(col_min, row_min)

        with rio.open(
            output_path,
            'w',
            driver='GTiff',
            height=the_slice.shape[0],
            width=the_slice.shape[1],
            count=1,
            dtype=the_slice.dtype, 
            crs=country_raster_1km_.crs,
            transform=new_transform,
            compress='lzw',
        ) as dst:
            dst.write(the_slice, 1)

        range_index_mammals.loc[i, ['row_min', 'row_max', 'col_min', 'col_max']] = [row_min, row_max, col_min, col_max]
        range_index_mammals.to_csv(r'../results/species-range-rasters/range-index-mammals.csv', index=False)
        
    except:
        pass


100%|██████████| 5750/5750 [2:15:58<00:00,  1.42s/it]  
